# Validate and publish Gold

Rebuild the complete release, review controls and coverage exceptions, and optionally publish a versioned Gold Delta release to MinIO. This notebook runs independently of notebooks 11–13.

Run from the `gold/` directory after setting `silver_manifest` in `gold_settings.json`.

In [ ]:
import json
from pathlib import Path

from IPython.display import display
import pandas as pd

import gold_pipeline
from gold_pipeline import build, publish

SETTINGS_PATH = Path(gold_pipeline.__file__).resolve().with_name("gold_settings.json")
settings_snapshot = SETTINGS_PATH.read_bytes()
configured_manifest = json.loads(settings_snapshot.decode("utf-8-sig")).get("silver_manifest")
print("Gold settings:", SETTINGS_PATH)
print("Silver manifest:", configured_manifest)
gold = build(SETTINGS_PATH, stage="products")
print("Silver release", gold.manifest["model_run_id"])
print("Cutoff", gold.settings["cutoff"])
display(pd.DataFrame([{"table": name, "rows": len(rows)} for name, rows in gold.tables.items()]))

In [ ]:
display(pd.DataFrame(gold.checks))
display(pd.DataFrame(gold.exceptions))

In [ ]:
from gold_delta import publish_to_minio

failed = [row for row in gold.checks if not row["passed"]]
if failed:
    raise ValueError(f"{len(failed)} Gold checks failed. Review the control results above.")

PUBLISH_TARGET = "preview"  # Set to "minio" to commit the validated release.
if PUBLISH_TARGET in {"minio", "local"} and SETTINGS_PATH.read_bytes() != settings_snapshot:
    raise ValueError("Gold settings changed after the build. Rerun the build cell before publishing.")
if PUBLISH_TARGET == "minio":
    gold_manifest_uri = publish_to_minio(gold)
    print("Gold manifest:", gold_manifest_uri)
elif PUBLISH_TARGET == "local":
    output_root = SETTINGS_PATH.resolve().parent / gold.settings["output_root"]
    print((publish(gold, output_root).resolve() / "manifest.json").as_posix())
elif PUBLISH_TARGET == "preview":
    print('Preview complete. Set PUBLISH_TARGET = "minio" to publish Gold Delta tables.')
else:
    raise ValueError('Use "preview", "minio", or "local".')